# Task 3 — Gender saved-model precision check

Use a **fresh Colab L4 runtime**, then Run All. This notebook reloads the same four saved models and the same 160-image samples per partition from diagnostic `20260905T083631345679Z`. It does not train. Publish this notebook and its new source module to the selected branch before running.


## 1. Connect Drive and load the repository


In [1]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import time
import zipfile

REPO_URL = "https://github.com/TrnLin/MLA2.git"
BRANCH = "fashion-analysis-and-cleanup"
CHECKOUT_DIR = Path("/content/MLA2")
DRIVE_MOUNT = Path("/content/drive")
DRIVE_PROJECT_DIR = DRIVE_MOUNT / "MyDrive/MLA2"
DATA_ZIP = DRIVE_PROJECT_DIR / "data/task3-data.zip"
LOCAL_DATA_ZIP = Path("/content/task3-data.zip")
DRIVE_TASK_DIR = DRIVE_PROJECT_DIR / "task3"
DRIVE_REGISTRY = DRIVE_TASK_DIR / "results/runs.csv"

def run_checked(command, *, cwd=None):
    command = [str(part) for part in command]
    print("$", " ".join(command), flush=True)
    return subprocess.run(command, cwd=cwd, check=True)

try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError("Connect this notebook to a Google Colab GPU runtime first.") from exc

drive.mount(str(DRIVE_MOUNT), force_remount=False)
if (CHECKOUT_DIR / ".git").is_dir():
    remote_url = subprocess.check_output(
        ["git", "remote", "get-url", "origin"], cwd=CHECKOUT_DIR, text=True
    ).strip()
    if remote_url != REPO_URL:
        raise RuntimeError(f"{CHECKOUT_DIR} belongs to a different repository: {remote_url}")
    run_checked(["git", "fetch", "origin", BRANCH], cwd=CHECKOUT_DIR)
    run_checked(["git", "switch", BRANCH], cwd=CHECKOUT_DIR)
    run_checked(["git", "merge", "--ff-only", f"origin/{BRANCH}"], cwd=CHECKOUT_DIR)
elif CHECKOUT_DIR.exists():
    raise RuntimeError(f"{CHECKOUT_DIR} exists but is not a Git repository.")
else:
    run_checked(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, CHECKOUT_DIR])

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=CHECKOUT_DIR, text=True).strip()
REPO_DIR = CHECKOUT_DIR / "core" if (CHECKOUT_DIR / "core/src/fashion").is_dir() else CHECKOUT_DIR
LOCAL_REGISTRY = REPO_DIR / "results/runs.csv"
print("Repository ready:", REPO_DIR)
print("Commit:", commit)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
$ git fetch origin task-3-gender-usage-classification
$ git switch task-3-gender-usage-classification
$ git merge --ff-only origin/task-3-gender-usage-classification
Repository ready: /content/MLA2
Commit: 5e66ce38fca1c17557ff30c4a7a58688a5e8381e


## 2. Restore images and the canonical split


In [2]:
def copy_teacher_zip_to_local_disk():
    if LOCAL_DATA_ZIP.is_file():
        try:
            with zipfile.ZipFile(LOCAL_DATA_ZIP) as existing:
                existing.infolist()
            return
        except zipfile.BadZipFile:
            LOCAL_DATA_ZIP.unlink()
    partial = LOCAL_DATA_ZIP.with_suffix(".zip.partial")
    for attempt in range(1, 4):
        partial.unlink(missing_ok=True)
        try:
            expected_bytes = DATA_ZIP.stat().st_size
            with DATA_ZIP.open("rb") as source, partial.open("wb") as target:
                shutil.copyfileobj(source, target, length=8 * 1024**2)
            if partial.stat().st_size != expected_bytes:
                raise OSError("The local ZIP copy is incomplete.")
            partial.replace(LOCAL_DATA_ZIP)
            return
        except OSError as error:
            partial.unlink(missing_ok=True)
            if attempt == 3:
                raise RuntimeError("Drive disconnected three times. Remount and retry.") from error
            drive.mount(str(DRIVE_MOUNT), force_remount=True)
            time.sleep(2)

copy_teacher_zip_to_local_disk()
teacher_dir = REPO_DIR / "data/raw/teacher"
required_files = (
    teacher_dir / "train/styles_train.csv",
    teacher_dir / "test/styles_prediction.csv",
)
image_dirs = (teacher_dir / "train/images_train", teacher_dir / "test/images_test")
image_suffixes = {".jpg", ".jpeg"}
with zipfile.ZipFile(LOCAL_DATA_ZIP) as archive:
    names = archive.namelist()
    if any(Path(name).is_absolute() or ".." in Path(name).parts for name in names):
        raise RuntimeError("The teacher archive contains an unsafe path.")
    expected_images = sum(
        name.startswith("data/raw/teacher/") and Path(name).suffix.lower() in image_suffixes
        for name in names
    )
    current_images = sum(
        path.suffix.lower() in image_suffixes for folder in image_dirs for path in folder.glob("*")
    )
    if current_images != expected_images or not all(path.is_file() for path in required_files):
        archive.extractall(REPO_DIR)

actual_images = sum(
    path.suffix.lower() in image_suffixes for folder in image_dirs for path in folder.glob("*")
)
if actual_images != expected_images or not all(path.is_file() for path in required_files):
    raise RuntimeError(f"Teacher data is incomplete: {actual_images:,}/{expected_images:,} images")

os.chdir(REPO_DIR)
os.environ["FASHION_PROJECT_ROOT"] = str(REPO_DIR)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
(DRIVE_TASK_DIR / "results").mkdir(parents=True, exist_ok=True)
print(f"Teacher data ready: {actual_images:,} images")


Teacher data ready: 44,441 images


## 3. Compare GPU number precision

The earlier diagnostic reproduced the full training and validation scores, but 64 of 72 sampled batch/order checks exceeded probability tolerance 0.00001. No sampled label changed.

Prepare each sample image once. Test the exact same tensors at batches 1, 32 and 128 in the original, reversed and shuffled orders. First record the current runtime settings; then request IEEE FP32 for matrix multiplication and convolution (TF32 disabled). Restore the original settings afterward. The earlier run did not record these precision flags, so check whether the new baseline reproduces its saved summaries before attributing the drift to precision.

Retain the 0.00001 tolerance and zero-label-change condition. Save all probability arrays, IDs, settings, hashes, model-state checks and peak GPU memory. Stop if model state changes or allocated GPU memory reaches 3,000,000,000 bytes.

Precision controls follow the [PyTorch 2.11 CUDA documentation](https://docs.pytorch.org/docs/2.11/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices). This notebook uses only the newer precision API.


In [3]:
from datetime import datetime, timezone
from fashion.train.task3_gender_precision import run_gender_precision

PRIOR = DRIVE_TASK_DIR / "diagnostics/gender_generalization/20260905T083631345679Z"
OUTPUT = DRIVE_TASK_DIR / "diagnostics/gender_precision" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
result = run_gender_precision(
    diagnostic_directory=PRIOR,
    g2_directory=DRIVE_TASK_DIR / "experiments/t3_gender_v2_g2_translation/gender",
    compact_directory=DRIVE_TASK_DIR / "experiments/t3_gender_e5_compact_blur_cnn/gender",
    registry_path=DRIVE_REGISTRY, output=OUTPUT, root=REPO_DIR,
)
print("Status:", result["status"])
print("Earlier summaries reproduced:", result["earlier_summaries_reproduced"])
print("All IEEE comparisons pass:", result["ieee_all_comparisons_pass"])
print("Saved:", OUTPUT)


Precision check: G2, fold 0
Precision check: G2, fold 4
Precision check: CompactBlurCNN, fold 0
Precision check: CompactBlurCNN, fold 4
Status: complete_for_review
Earlier summaries reproduced: True
All IEEE comparisons pass: True
Saved: /content/drive/MyDrive/MLA2/task3/diagnostics/gender_precision/20260905T085822668071Z


## 4. Stop for review

Tell me when this finishes. Read `precision_status.json`, `precision_comparisons.csv`, and each model's probability archives. The saved arrays allow independent checks of every reported delta and label change.

Completion is not a passing decision. Retain the original diagnostic's `review_required` status. If the baseline fails to reproduce or drift remains, review the settings and results. Even if IEEE FP32 removes the sampled drift, that alone does not prove a narrower model will help. Do not start training here.
